# 01 — NOAA Storm Events Loading
## Why NOAA Storm Events?
NOAA's Storm Events Database records individual weather events with physical intensity
metrics (wind speed, precipitation magnitude, property damage estimates) at the county level.
This is the key missing feature from Phase 1 — we have WHERE and WHAT TYPE the disaster was,
but not HOW SEVERE the specific event was physically.

## Join strategy
NOAA events are matched to FEMA declarations using:
- State FIPS code (NOAA STATE_FIPS → FEMA stateNumberCode)
- County FIPS code (NOAA CZ_FIPS → FEMA countyCode)  
- Date overlap: NOAA event dates overlap with FEMA incident begin/end dates

In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import gzip
import io
import warnings
warnings.filterwarnings('ignore')

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROCESSED, exist_ok=True)

YEARS = list(range(1998, 2024))  # match FEMA data range

## Download NOAA Storm Events Detail Files
We scrape the NOAA directory listing to find the correct filename for each year,
then download and cache the gzip-compressed CSV files locally.

In [ ]:
import urllib.request
from bs4 import BeautifulSoup

BASE_URL = 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/'

def get_noaa_filenames(year):
    """Scrape NOAA directory to find the detail file for a given year."""
    resp = requests.get(BASE_URL, timeout=30)
    soup = BeautifulSoup(resp.text, 'html.parser')
    links = [a['href'] for a in soup.find_all('a', href=True)]
    detail_files = [l for l in links if f'details' in l and f'd{year}_' in l and l.endswith('.csv.gz')]
    return detail_files

def download_noaa_year(year):
    """Download and return NOAA Storm Events detail CSV for a year."""
    out_path = RAW + f'noaa_storm_events_{year}.csv'
    if os.path.exists(out_path):
        return pd.read_csv(out_path, low_memory=False)
    
    files = get_noaa_filenames(year)
    if not files:
        print(f'  No file found for {year}, skipping')
        return None
    
    url = BASE_URL + files[0]
    print(f'  Downloading {year}: {files[0]}')
    resp = requests.get(url, timeout=60)
    with gzip.open(io.BytesIO(resp.content)) as f:
        df = pd.read_csv(f, low_memory=False)
    df.to_csv(out_path, index=False)
    return df

frames = []
for yr in YEARS:
    print(f'Year {yr}...')
    df = download_noaa_year(yr)
    if df is not None:
        frames.append(df)

noaa_raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal NOAA rows: {len(noaa_raw):,}')
print(f'Columns: {list(noaa_raw.columns)}')

## Filter to Disaster-Relevant Event Types
NOAA Storm Events includes many event types (e.g., Waterspout, Dust Devil) that rarely
result in FEMA declarations. We keep only event types that have meaningful overlap with
declared federal disasters. We also parse the DAMAGE columns (which use K/M/B suffixes)
into numeric USD values and standardize FIPS codes.

In [ ]:
# Keep only event types that map to FEMA disaster declarations
RELEVANT_TYPES = [
    'Hurricane', 'Hurricane (Typhoon)', 'Tropical Storm',
    'Flood', 'Flash Flood', 'Coastal Flood', 'Lakeshore Flood',
    'Tornado', 'Thunderstorm Wind', 'High Wind', 'Strong Wind',
    'Winter Storm', 'Ice Storm', 'Blizzard', 'Heavy Snow',
    'Wildfire', 'Drought', 'Earthquake', 'Tsunami',
    'Storm Surge/Tide', 'Heavy Rain'
]

noaa = noaa_raw[noaa_raw['EVENT_TYPE'].isin(RELEVANT_TYPES)].copy()

# Parse damage columns (NOAA uses K/M/B suffixes)
def parse_damage(val):
    if pd.isna(val) or val == '':
        return 0.0
    val = str(val).strip().upper()
    multipliers = {'K': 1e3, 'M': 1e6, 'B': 1e9}
    if val[-1] in multipliers:
        return float(val[:-1]) * multipliers[val[-1]]
    try:
        return float(val)
    except:
        return 0.0

noaa['damage_property_usd'] = noaa['DAMAGE_PROPERTY'].apply(parse_damage)
noaa['damage_crops_usd']    = noaa['DAMAGE_CROPS'].apply(parse_damage)
noaa['total_damage_usd']    = noaa['damage_property_usd'] + noaa['damage_crops_usd']

# Standardise FIPS
noaa['state_fips']  = noaa['STATE_FIPS'].astype(str).str.zfill(2)
noaa['county_fips'] = noaa['CZ_FIPS'].astype(str).str.zfill(3)
noaa['fips']        = noaa['state_fips'] + noaa['county_fips']

# Parse dates
noaa['begin_date'] = pd.to_datetime(noaa['BEGIN_DATE_TIME'], errors='coerce')
noaa['end_date']   = pd.to_datetime(noaa['END_DATE_TIME'],   errors='coerce')
noaa['year']       = noaa['begin_date'].dt.year

KEEP_COLS = ['fips', 'state_fips', 'year', 'begin_date', 'end_date',
             'EVENT_TYPE', 'MAG', 'MAGNITUDE_TYPE',
             'INJURIES_DIRECT', 'INJURIES_INDIRECT',
             'DEATHS_DIRECT', 'DEATHS_INDIRECT',
             'damage_property_usd', 'damage_crops_usd', 'total_damage_usd',
             'BEGIN_LAT', 'BEGIN_LON']

noaa = noaa[[c for c in KEEP_COLS if c in noaa.columns]]
noaa.columns = [c.lower() for c in noaa.columns]

print(f'Filtered NOAA rows: {len(noaa):,}')
print(f'Event types kept: {noaa["event_type"].nunique()}')
print(noaa.dtypes)

## Save Filtered Dataset
Save the filtered NOAA data to processed for use in notebook 02.

In [ ]:
noaa.to_csv(PROCESSED + 'noaa_filtered.csv', index=False)
print(f'Saved: noaa_filtered.csv ({len(noaa):,} rows)')